# Módulo 2 – EDA y Calidad de Datos (Google Colab)
Diplomado de Grado – **UNICOLOMBO**
**Autor:** Heyder Medrano Olier – Docente Investigador
**Objetivo:** Diagnosticar, limpiar e integrar datos y exportar un CSV listo para BI.

### Bibliografía
- McKinney, W. (2022). *Python for Data Analysis*. O’Reilly.
- Wickham, H. (2023). *R for Data Science (2nd ed.)*. O’Reilly.
- pandas docs: https://pandas.pydata.org/docs/  
- Power BI docs: https://learn.microsoft.com/power-bi/


In [0]:
# !pip install pandas numpy matplotlib seaborn plotly openpyxl
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
pd.set_option('display.max_columns', 60)


In [0]:
ventas = pd.read_csv('Clientes_Ventas_Calidad_Datos.csv')
clientes = pd.read_csv('Clientes_Info.csv')
ventas.shape, clientes.shape

In [0]:
ventas['Fecha'] = pd.to_datetime(ventas['Fecha'], errors='coerce', dayfirst=True)
clientes['Fecha_Alta'] = pd.to_datetime(clientes['Fecha_Alta'], errors='coerce', dayfirst=True)
ventas.isnull().mean().sort_values(ascending=False).head(10)

In [0]:
num_cols_v = ventas.select_dtypes(include=['float64','int64']).columns
cat_cols_v = ventas.select_dtypes(include=['object']).columns
ventas[num_cols_v] = ventas[num_cols_v].apply(lambda s: s.fillna(s.median()))
for c in cat_cols_v:
    ventas[c] = ventas[c].fillna(ventas[c].mode().iloc[0] if not ventas[c].mode().empty else 'Desconocido')
ventas = ventas.drop_duplicates()
num_cols_c = clientes.select_dtypes(include=['float64','int64']).columns
cat_cols_c = clientes.select_dtypes(include=['object']).columns
clientes[num_cols_c] = clientes[num_cols_c].apply(lambda s: s.fillna(s.median()))
for c in cat_cols_c:
    clientes[c] = clientes[c].fillna(clientes[c].mode().iloc[0] if not clientes[c].mode().empty else 'Desconocido')
clientes = clientes.drop_duplicates()
ventas.shape, clientes.shape

In [0]:
def iqr_bounds(s):
    q1,q3 = s.quantile([0.25,0.75]); iqr = q3-q1
    return q1-1.5*iqr, q3+1.5*iqr
low_v, high_v = iqr_bounds(ventas['Venta_Total'])
ventas['Outlier_Venta'] = ~ventas['Venta_Total'].between(low_v, high_v)
sns.boxplot(x='Categoría', y='Venta_Total', data=ventas.sample(6000, random_state=1)); plt.xticks(rotation=45); plt.show()

In [0]:
merged = ventas.merge(clientes, on='Cliente_ID', how='left', suffixes=('','_Cliente'))
with pd.ExcelWriter('Datos_Limpiados_Modulo2.xlsx', engine='openpyxl') as w:
    ventas.to_excel(w, 'Ventas_Limpias', index=False)
    clientes.to_excel(w, 'Clientes_Limpios', index=False)
merged.to_csv('Ventas_Clientes_Limpio.csv', index=False)
print('Exportado: Datos_Limpiados_Modulo2.xlsx y Ventas_Clientes_Limpio.csv')